# Entrega H2 — Proyecto RecSys
## Tema: Multimodalidad — Recomendación de Videojuegos

**Dataset:** [Game Recommendations on Steam](https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam)

En H1 se establecieron cuatro baselines: Random, Most Popular, CB-TF-IDF y ALS.  
En H2 se incorpora **BPR (Bayesian Personalized Ranking)** como nuevo modelo colaborativo que optimiza ranking pairwise directamente, más alineado con la tarea top-K que ALS.

La evaluación se extiende a K ∈ {5, 10, 20} e incluye métricas de diversidad: Coverage, Novelty e ILD (Intra-List Diversity).

In [ ]:
!pip install -q kagglehub implicit

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
from IPython.display import display
import implicit
import kagglehub

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE  = 42
KS            = (5, 10, 20)
TOP_K         = max(KS)
MAX_EVAL      = 2_000
ALPHA         = 40
MIN_USER      = 5
MIN_GAME      = 20
SAMPLE_FRAC   = 0.10

## 1. Carga de datos

Se carga `recommendations.csv` (~41 M filas) en chunks con tipos de dato optimizados para reducir uso de memoria.

In [ ]:
PATH = kagglehub.dataset_download('antonkozyriev/game-recommendations-on-steam')
print('Archivos:', os.listdir(PATH))

chunks = []
for chunk in pd.read_csv(
    os.path.join(PATH, 'recommendations.csv'),
    chunksize=1_000_000,
    dtype={'app_id': 'int32', 'user_id': 'int32',
           'hours': 'float32', 'is_recommended': 'bool',
           'helpful': 'int16', 'funny': 'int16'},
    parse_dates=['date']
):
    chunks.append(chunk)
recs = pd.concat(chunks, ignore_index=True)
print(f'Cargado: {len(recs):,} filas — {recs.memory_usage(deep=True).sum()/1e9:.2f} GB')

## 2. Preprocesamiento

Se replican exactamente los pasos del H1:
1. **Deduplicación** — por par (user, juego), manteniendo la interacción más reciente.
2. **Señal de confianza** — `confidence = log1p(hours)` para interacciones positivas.
3. **K-core iterativo** — `min_user=5`, `min_game=20`; converge en 9 iteraciones.
4. **Subsample 10%** de usuarios para tractabilidad.
5. **Split temporal leave-one-out** — última interacción de cada usuario como test; solo se evalúan los usuarios cuyo ítem de test es positivo.

In [ ]:
# 1. Deduplicación
recs = (recs.sort_values('date')
            .drop_duplicates(subset=['user_id', 'app_id'], keep='last')
            .reset_index(drop=True))
print(f'Post-dedup: {len(recs):,}')

# 2. Señal de confianza
recs['confidence'] = np.where(
    recs['is_recommended'],
    np.log1p(recs['hours'].clip(0)),
    0.0
)

# 3. K-core iterativo
def k_core(df, min_u=MIN_USER, min_g=MIN_GAME):
    for it in range(1, 30):
        n0 = len(df)
        uc = df['user_id'].value_counts()
        gc = df['app_id'].value_counts()
        df = df[
            df['user_id'].isin(uc[uc >= min_u].index) &
            df['app_id'].isin(gc[gc >= min_g].index)
        ].copy()
        print(f'  iter {it}: {n0:,} → {len(df):,}')
        if len(df) == n0:
            break
    return df

print('K-core filtering...')
recs = k_core(recs)
print(f'Post k-core: {recs["user_id"].nunique():,} usuarios | {recs["app_id"].nunique():,} juegos')

# 4. Subsample del 10% de usuarios
np.random.seed(RANDOM_STATE)
sampled = pd.Series(recs['user_id'].unique()).sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
recs = recs[recs['user_id'].isin(sampled)].copy()
print(f'Submuestra: {len(recs):,} interacciones | {recs["user_id"].nunique():,} usuarios')

# 5. Split temporal leave-one-out
recs = recs.sort_values(['user_id', 'date'])
last_idx = recs.groupby('user_id').tail(1).index
train_df = recs.drop(index=last_idx).reset_index(drop=True)
test_df  = recs.loc[last_idx].reset_index(drop=True)

test_pos = test_df[test_df['is_recommended']].copy()
test_item_per_user   = dict(zip(test_pos['user_id'], test_pos['app_id']))
train_items_per_user = train_df.groupby('user_id')['app_id'].apply(set).to_dict()

print(f'Train: {len(train_df):,} | Test evaluable: {len(test_pos):,} usuarios ({len(test_pos)/len(test_df)*100:.1f}%)')

In [ ]:
user_list   = sorted(train_df['user_id'].unique())
item_list   = sorted(recs['app_id'].unique())
user_to_idx = {u: i for i, u in enumerate(user_list)}
item_to_idx = {a: j for j, a in enumerate(item_list)}
n_users     = len(user_list)
n_items     = len(item_list)

# Interacciones positivas de train: base para construir matrices ALS y BPR
tp   = train_df[train_df['is_recommended']].copy()
tp   = tp[tp['user_id'].isin(user_to_idx) & tp['app_id'].isin(item_to_idx)]
rows = tp['user_id'].map(user_to_idx).to_numpy()
cols = tp['app_id'].map(item_to_idx).to_numpy()

# Popularidad para fallback (cold-start) y métrica de novedad
popularity    = train_df[train_df['is_recommended']].groupby('app_id').size()
popular_list  = popularity.sort_values(ascending=False).index.tolist()
item_pop_dict = popularity.to_dict()

# Usuarios evaluables (muestra de máx MAX_EVAL)
rng        = np.random.default_rng(RANDOM_STATE)
eval_users = list(test_item_per_user.keys())
if len(eval_users) > MAX_EVAL:
    eval_users = list(rng.choice(eval_users, MAX_EVAL, replace=False))

n_train_users = train_df['user_id'].nunique()
print(f'n_users={n_users:,} | n_items={n_items:,} | eval_users={len(eval_users):,}')

## 3. Framework de evaluación

Se extiende el H1 de dos formas:
- **Múltiples K**: Precision, Recall y NDCG evaluados en K ∈ {5, 10, 20}.
- **Métricas de diversidad**: Coverage@10, Novelty@10 e ILD@10 (Intra-List Diversity usando factores latentes del modelo).
- **Desagregación por actividad**: NDCG@10 separado por cantidad de interacciones en train del usuario.

In [ ]:
def precision_at_k(rec, rel, k):
    return sum(1 for x in rec[:k] if x in rel) / k

def recall_at_k(rec, rel, k):
    hits = sum(1 for x in rec[:k] if x in rel)
    return hits / len(rel) if rel else 0.0

def ndcg_at_k(rec, rel, k):
    dcg  = sum(1 / np.log2(i + 2) for i, x in enumerate(rec[:k]) if x in rel)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_at_ks(recs, test_dict, ks=KS):
    scores = {k: {'P': [], 'R': [], 'N': []} for k in ks}
    for u, rec in recs.items():
        if u not in test_dict:
            continue
        rel = {test_dict[u]}
        for k in ks:
            scores[k]['P'].append(precision_at_k(rec, rel, k))
            scores[k]['R'].append(recall_at_k(rec, rel, k))
            scores[k]['N'].append(ndcg_at_k(rec, rel, k))
    result = {}
    for k in ks:
        result[f'Precision@{k}'] = float(np.mean(scores[k]['P']))
        result[f'Recall@{k}']    = float(np.mean(scores[k]['R']))
        result[f'NDCG@{k}']      = float(np.mean(scores[k]['N']))
    result['n_users'] = len(scores[ks[0]]['P'])
    return result

def coverage_at_k(recs, catalog_size, k=10):
    return len({x for items in recs.values() for x in items[:k]}) / catalog_size

def novelty_at_k(recs, pop_dict, n_train, k=10):
    scores = []
    for items in recs.values():
        nov = [-np.log2(pop_dict.get(i, 1) / n_train + 1e-10) for i in items[:k]]
        scores.append(float(np.mean(nov)))
    return float(np.mean(scores)) if scores else 0.0

def ild_at_k(recs, i2idx, factors, k=10):
    """Intra-List Diversity = 1 - avg cosine sim entre pares en top-K."""
    f = np.array(factors, dtype='float32')
    scores = []
    for items in recs.values():
        idxs = [i2idx[i] for i in items[:k] if i in i2idx]
        if len(idxs) < 2:
            continue
        vecs    = normalize(f[idxs])
        sim     = vecs @ vecs.T
        n       = len(idxs)
        avg_sim = (sim.sum() - n) / (n * (n - 1))
        scores.append(1.0 - float(avg_sim))
    return float(np.mean(scores)) if scores else 0.0

def activity_ndcg(recs, test_dict, train_dict, k=10):
    bins = [(2, 5, '2-5'), (6, 10, '6-10'), (11, 20, '11-20'), (21, 9999, '21+')]
    result = {}
    for lo, hi, label in bins:
        vals = [
            ndcg_at_k(rec, {test_dict[u]}, k)
            for u, rec in recs.items()
            if u in test_dict and lo <= len(train_dict.get(u, set())) <= hi
        ]
        result[label] = (float(np.mean(vals)) if vals else 0.0, len(vals))
    return result

print('Framework de evaluación listo.')

## 4. Baseline: Most Popular

Recomienda los juegos con más interacciones positivas en entrenamiento, excluyendo los ya vistos por el usuario. Referencia del H1: NDCG@10 = 0.0222.

In [ ]:
recs_pop = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    recs_pop[u] = [i for i in popular_list if i not in seen][:TOP_K]

metrics_pop = evaluate_at_ks(recs_pop, test_item_per_user)
print('Most Popular')
for k in KS:
    print(f'  P@{k}={metrics_pop[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_pop[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_pop[f"NDCG@{k}"]:.4f}')

## 5. ALS — Filtrado Colaborativo (referencia H1)

ALS con feedback implícito. Minimiza el error cuadrático ponderado por la confianza $c_{ui} = 1 + \alpha \cdot \log(1 + \text{hours})$ con $\alpha=40$. Se incluye para comparación directa con BPR manteniendo los mismos hiperparámetros del H1: `factors=64`, `regularization=0.1`, `iterations=15`.

In [ ]:
conf_data = (1.0 + ALPHA * tp['confidence'].to_numpy()).astype('float32')
user_item = csr_matrix((conf_data, (rows, cols)), shape=(n_users, n_items))

als_model = implicit.als.AlternatingLeastSquares(
    factors=64, regularization=0.1, iterations=15,
    calculate_training_loss=True, random_state=RANDOM_STATE
)
als_model.fit(user_item, show_progress=True)

recs_als = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    if u not in user_to_idx:
        recs_als[u] = [i for i in popular_list if i not in seen][:TOP_K]
        continue
    ui = user_to_idx[u]
    ids, _ = als_model.recommend(
        ui, user_item[ui], N=TOP_K + len(seen), filter_already_liked_items=True
    )
    recs_als[u] = [item_list[j] for j in ids if item_list[j] not in seen][:TOP_K]

metrics_als = evaluate_at_ks(recs_als, test_item_per_user)
div_als = {
    'Coverage@10': coverage_at_k(recs_als, n_items),
    'Novelty@10':  novelty_at_k(recs_als, item_pop_dict, n_train_users),
    'ILD@10':      ild_at_k(recs_als, item_to_idx, als_model.item_factors),
}
print('ALS')
for k in KS:
    print(f'  P@{k}={metrics_als[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_als[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_als[f"NDCG@{k}"]:.4f}')
for m, v in div_als.items():
    print(f'  {m}={v:.4f}')

## 6. BPR — Bayesian Personalized Ranking

BPR optimiza directamente el ranking pairwise: para cada usuario $u$, un ítem positivo observado $j$ debe estar mejor rankeado que cualquier ítem no observado $k$:

$$\text{BPR-OPT} = \sum_{(u,j,k) \in D_S} \ln \sigma(\hat{x}_{ujk}) - \lambda \|\Theta\|^2$$

donde $\hat{x}_{ujk} = \hat{x}_{uj} - \hat{x}_{uk}$ es la diferencia de scores y $\sigma$ es la sigmoide.

**Diferencias clave con ALS:**

| | ALS | BPR |
|---|---|---|
| **Objetivo** | Minimiza MSE ponderado por confianza | Maximiza log-verosimilitud del ranking |
| **Señal de entrada** | Matriz de confianza: $1 + 40 \cdot \log(1+h)$ | Matriz binaria 0/1 |
| **Optimización** | Cuadrados mínimos alternados (batch) | SGD sobre triples $(u, j^+, k^-)$ |
| **Alineación top-K** | Indirecta | Directa |

Se usan los mismos `factors=64` que ALS para comparación justa. BPR requiere más iteraciones (100 vs 15) porque SGD converge más lentamente que el método batch de ALS.

In [ ]:
try:
    from implicit.bpr import BayesianPersonalizedRanking
except ImportError:
    from implicit.cpu.bpr import BayesianPersonalizedRanking

# Matriz binaria: BPR optimiza ranking relativo, no intensidad de preferencia
user_item_bpr = csr_matrix(
    (np.ones(len(tp), dtype='float32'), (rows, cols)),
    shape=(n_users, n_items)
)

bpr_model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.01,
    iterations=100,
    random_state=RANDOM_STATE,
)
bpr_model.fit(user_item_bpr, show_progress=True)

recs_bpr = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    if u not in user_to_idx:
        recs_bpr[u] = [i for i in popular_list if i not in seen][:TOP_K]
        continue
    ui = user_to_idx[u]
    ids, _ = bpr_model.recommend(
        ui, user_item_bpr[ui], N=TOP_K + len(seen), filter_already_liked_items=True
    )
    recs_bpr[u] = [item_list[j] for j in ids if item_list[j] not in seen][:TOP_K]

metrics_bpr = evaluate_at_ks(recs_bpr, test_item_per_user)
div_bpr = {
    'Coverage@10': coverage_at_k(recs_bpr, n_items),
    'Novelty@10':  novelty_at_k(recs_bpr, item_pop_dict, n_train_users),
    'ILD@10':      ild_at_k(recs_bpr, item_to_idx, bpr_model.item_factors),
}
print('BPR')
for k in KS:
    print(f'  P@{k}={metrics_bpr[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_bpr[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_bpr[f"NDCG@{k}"]:.4f}')
for m, v in div_bpr.items():
    print(f'  {m}={v:.4f}')

## 7. Comparación de resultados

In [ ]:
# ── Tabla completa de métricas ──
model_info = [
    ('Most Popular', recs_pop, metrics_pop, None),
    ('ALS',          recs_als, metrics_als, als_model.item_factors),
    ('BPR',          recs_bpr, metrics_bpr, bpr_model.item_factors),
]

rows_table = []
for name, recs_m, met, factors in model_info:
    row = {'Modelo': name}
    for k in KS:
        row[f'P@{k}']    = met[f'Precision@{k}']
        row[f'R@{k}']    = met[f'Recall@{k}']
        row[f'NDCG@{k}'] = met[f'NDCG@{k}']
    row['Coverage@10'] = coverage_at_k(recs_m, n_items)
    row['Novelty@10']  = novelty_at_k(recs_m, item_pop_dict, n_train_users)
    row['ILD@10']      = ild_at_k(recs_m, item_to_idx, factors) if factors is not None else float('nan')
    rows_table.append(row)

df_results = pd.DataFrame(rows_table).set_index('Modelo')
print('=== Métricas de Ranking y Diversidad ===')
display(df_results.round(4))

# ── Gráfico NDCG@K ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#3498DB', '#E67E22', '#27AE60']
names  = [name for name, *_ in model_info]
for ax, k in zip(axes, KS):
    vals = [met[f'NDCG@{k}'] for _, _, met, _ in model_info]
    bars = ax.bar(names, vals, color=colors, alpha=0.85, edgecolor='black')
    ax.set_title(f'NDCG@{k}', fontsize=13)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.suptitle(f'Comparación de modelos (n={len(eval_users):,} usuarios)', fontsize=13)
plt.tight_layout()
plt.savefig('ndcg_h2_bpr.png', bbox_inches='tight')
plt.show()

# ── Desagregación por nivel de actividad ──
print('\n=== NDCG@10 por nivel de actividad (nº interacciones en train) ===')
act_bins  = ['2-5', '6-10', '11-20', '21+']
act_table = {}
counts    = {}
for name, recs_m, _, _ in model_info:
    act = activity_ndcg(recs_m, test_item_per_user, train_items_per_user)
    act_table[name] = {label: val for label, (val, _) in act.items()}
    if not counts:
        counts = {label: cnt for label, (_, cnt) in act.items()}

df_act = pd.DataFrame(act_table).T[act_bins]
print(f'N usuarios por grupo: {counts}')
display(df_act.round(4))

# ── Mejoras relativas vs Most Popular ──
print('\nMejora de NDCG@10 relativa a Most Popular:')
base = metrics_pop['NDCG@10']
for name, _, met, _ in model_info:
    val   = met['NDCG@10']
    delta = (val / max(base, 1e-9) - 1) * 100
    print(f'  {name:<15}: {val:.4f}  ({delta:+.1f}%)')